# AI Herding · Next Step 4: Fifteen-round comparisons with shared dividends

**A complete Jupyter notebook with 9 code cells, using Step 1's size cap and Step 2's order observer.**

Step 4 can be prepared before the new Step 3 model experiment is complete. It examines whether explicit two-sided quoting continues to produce executed trades in the simulated market and how recorded prices, execution prices, and two-sided midpoints evolve.

| Mode | Procedure | New model API requests | Reportable progress |
|---|---|---|---|
| `prepare_only` (default) | Generate fixed dividend paths and the plan; check the 15-round workflow with explicitly scripted orders | 0 | Experimental preparation and offline workflow validation |
| `api_experiment` | All 8 agents request model responses from the updated market state each round in a formal 15-round experiment | 1,080 planned by default | New multi-round model results only after completion |

**Default mode produces no new model-experiment findings.** Its buy/sell decisions are manually scripted. Results stay in `offline_checks` and are excluded from formal result aggregation. No real model requests were made before delivery; the checks cover dividend injection, multi-round logging, matching, resource linkage, and repeated replay.

The default formal plan is three maker prompts × three dividend paths (seeds 42, 43, 44) × 15 rounds × 8 agents = **1,080 planned requests**. This excludes additional uncached attempts during manual resumption; the code does not retry automatically. Charges depend on the service and actual tokens; this notebook does not estimate cost.

Retain the original 56 scenario's infinite-horizon setting: pay a dividend each round of the 15-round observation window, with no terminal redemption in round 15. Fundamental value is approximately 28, and dividends are 0.4 or 2.4 per share. Conditions sharing a seed use **exactly the same realized dividend sequence**, unaffected by other random-number consumption. Future realizations are excluded from agent prompts.

Other agents may respond differently as market histories evolve. Fix dividends, initial conditions, and processing rules. This differs from Step 3's one-round protocol that freezes untreated-agent responses.

The default model remains `gpt-4o-2024-11-20`. Before a formal run, specify the model and endpoint confirmed by the team; there is no automatic model substitution. Use the same model settings across all three conditions. References: [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=chat) and [GPT-4o model page](https://developers.openai.com/api/docs/models/gpt-4o).


## Code cell 1/9: Select the mode and experiment size

Keep `prepare_only` for the first run. Only changing it to `api_experiment` enables real model requests in cell 7.

For a smaller formal pilot, set `SEEDS` to `[42]` and `ARMS` to `["current", "mandatory_two_sided"]`: 2 × 15 × 8 = 240 planned requests. A single-seed pilot does not replace repeated experiments. `MAX_NEW_API_REQUESTS` limits new requests in this execution.


In [ ]:
import sys,os,json,csv,hashlib,shutil,subprocess,ast,uuid,zipfile
from pathlib import Path
from datetime import datetime,timezone
from collections import defaultdict
from html import escape

RUN_MODE = "prepare_only"  # Select "api_experiment" for the formal experiment; this calls the model API.
RUN_OFFLINE_CHECK = True  # Check the workflow with scripted orders; no model calls or new model findings.
SEEDS = [42,43,44]
INITIAL_PRICES = [56.0]
ARMS = ["current","earliest_public","mandatory_two_sided"]
API_MODEL = "gpt-4o-2024-11-20"  # Use the same team-confirmed model in every condition; no automatic substitution.
API_BASE_URL = ""  # Leave blank for the OpenAI endpoint, or enter the complete team-compatible endpoint URL.
MAX_NEW_API_REQUESTS = 1080
STEP2_WORK_OVERRIDE = None

assert RUN_MODE in ("prepare_only","api_experiment")
assert SEEDS and all(isinstance(seed,int) for seed in SEEDS)
assert len(set(SEEDS))==len(SEEDS)
assert ARMS and set(ARMS)<={"current","earliest_public","mandatory_two_sided"}
if sys.version_info<(3,11):raise RuntimeError("Use Python 3.11 or later.")
hint=STEP2_WORK_OVERRIDE or os.environ.get("AHE_STEP2_WORKDIR")
candidates=[Path(hint).expanduser()] if hint else [Path.cwd()/"AI_Herding_Step2_Work",Path.home()/"AI_Herding_Step2_Work"]
if not hint:candidates += [p/"AI_Herding_Step2_Work" for p in list(Path.cwd().parents)[:3]]
STEP2_WORK=next((p.resolve() for p in candidates if (p/"source_logged/src/order_lifecycle_audit.py").is_file()),None)
if STEP2_WORK is None:raise FileNotFoundError("Set STEP2_WORK_OVERRIDE to the Step 2 AI_Herding_Step2_Work directory.")
SOURCE_LOGGED=STEP2_WORK/"source_logged"
EXPECTED_BASE_HASHES={'src/market/orders/order_state_manager.py': '9ed4404e4c54f0dfd32c4bff1b047264fdaab19b2eb3a1c47b15f60679a28020', 'src/market/orders/order_service_factory.py': '3fa6ae07a599b36a9f095a207cb082b6681ad3817bf0d7f633e4959b428586de', 'src/base_sim.py': 'b5dc6d64012d372be01771ca90385db9a3da282e693c03e97ce2936ab28b379d', 'src/order_lifecycle_audit.py': '274539040db6008a230a21b9682c7cade7f87488eb6b316602f1599ad57cc634'}
for relative,expected in EXPECTED_BASE_HASHES.items():
    if hashlib.sha256((SOURCE_LOGGED/relative).read_text(encoding="utf-8").encode()).hexdigest()!=expected:
        raise RuntimeError(f"Source from the previous steps contains other changes; inspect: {relative}")
TEST_PYTHON=STEP2_WORK.parent/"AI_Herding_Step1_Work/test_environment"/("Scripts/python.exe" if os.name=="nt" else "bin/python")
if not TEST_PYTHON.is_file():raise FileNotFoundError("Step 1 test environment not found. Keep the working directories from the previous steps.")
WORK_DIR=STEP2_WORK.parent/"AI_Herding_Step4_Work";WORK_DIR.mkdir(exist_ok=True)
SOURCE_ROOT=WORK_DIR/"source_longitudinal"
SESSION_DIR=WORK_DIR/("session_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:6])
SESSION_DIR.mkdir();PROTOCOL_DIR=SESSION_DIR/"protocol";PROTOCOL_DIR.mkdir()
PLANNED_REQUESTS=len(SEEDS)*len(INITIAL_PRICES)*len(ARMS)*15*8
print("Current mode: ",RUN_MODE)
print("Formal plan: ",len(SEEDS)*len(INITIAL_PRICES)*len(ARMS),"market runs; 15 rounds per run; ",PLANNED_REQUESTS,"model requests.")
print("Model requests in default preparation mode: 0.")
print("Output directory: ",SESSION_DIR)


## Code cell 2/9: Prepare dependencies and display helpers

Reuse Step 1's environment. Installing Python packages is not a model call. The offline check imports the original simulator and therefore needs its SDK dependency, but sends no model requests.


In [ ]:
probe_source = r"""
import importlib,json,sys
needed=[]
items=[('numpy','numpy>=1.26'),('pandas','pandas>=2.2'),('pydantic','pydantic>=2.11,<3')]
if sys.argv[1]=='yes':items += [('openai','openai>=1.86,<3'),('dotenv','python-dotenv>=1.1')]
for name,spec in items:
    try:importlib.import_module(name)
    except (ImportError,OSError):needed.append(spec)
print(json.dumps(needed))
"""
need_engine=RUN_OFFLINE_CHECK or RUN_MODE=="api_experiment"
probe=subprocess.run([str(TEST_PYTHON),"-c",probe_source,"yes" if need_engine else "no"],capture_output=True,text=True,encoding="utf-8",check=True)
needed=json.loads(probe.stdout)
if needed:
    install=subprocess.run([str(TEST_PYTHON),"-m","pip","install","--disable-pip-version-check",*needed],capture_output=True,text=True,encoding="utf-8")
    if install.returncode:raise RuntimeError(install.stdout[-2000:]+install.stderr[-3000:])
def read_json(path):return json.loads(Path(path).read_text(encoding="utf-8"))
def read_csv(path):
    with Path(path).open(encoding="utf-8-sig",newline="") as stream:return list(csv.DictReader(stream))
def display_table(rows,columns):
    from IPython.display import display,HTML
    head="".join(f"<th>{escape(str(label))}</th>" for key,label in columns)
    body="".join("<tr>"+"".join(f"<td>{escape(str(row.get(key,'')))}</td>" for key,label in columns)+"</tr>" for row in rows)
    display(HTML("<table border='1' cellpadding='6' style='border-collapse:collapse'><tr>"+head+"</tr>"+body+"</table>"))
def child_env():
    env=os.environ.copy();env['PYTHONIOENCODING']='utf-8';return env
def stream_command(command,env=None):
    process=subprocess.Popen(command,cwd=SOURCE_ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,encoding="utf-8",env=env or child_env())
    try:
        for line in process.stdout:print(line.rstrip(),flush=True)
        if process.wait()!=0:raise RuntimeError("The run did not complete; inspect the error above. Successfully cached model responses are retained.")
    except BaseException:
        if process.poll() is None:process.terminate();process.wait(timeout=15)
        raise
print("Dependencies ready; new model requests in this cell: 0.")


## Code cell 3/9: Write the multi-round experiment code

Write the runner and multi-round observer adapter into the Step 4 copy. Align decision and matching round indices and reconcile returned trades against IDs newly recorded during each matching call. Keep the prior source copies.

Use the original simulator's decision format, market information, resource checks, matching, dividend payments, and interest payments. Request agents in order 0–7 and process their submissions together each round. Stop on API errors or a fallback path rather than silently using an artificial Hold response.


In [ ]:
NEW_FILES = {
'scripts/run_step4_longitudinal.py': r"""'''Paired 15-round experiments with pre-generated dividends and explicit API mode.

The smoke mode uses declared scripted responses solely to verify the machinery.
It never supplies research results or response caches to the API experiment.
'''
from __future__ import annotations
import argparse
import csv
import hashlib
import json
import logging
import math
import os
import random
import socket
import sys
import uuid
from collections import Counter, defaultdict
from concurrent.futures import Future
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

ARMS=('current','earliest_public','mandatory_two_sided')
AGENTS=[str(i) for i in range(8)]
MAKERS={'4','5'}
VERSION='step4-v1'


def canonical(value):
    if isinstance(value,float) and not math.isfinite(value):return str(value)
    if isinstance(value,dict):return {str(k):canonical(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [canonical(v) for v in value]
    return value


def digest(value):
    return hashlib.sha256(json.dumps(canonical(value),sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()


def save_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(canonical(value),ensure_ascii=False,indent=2),encoding='utf-8')
    temp.replace(path)


def write_csv(path,rows):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    keys=list(dict.fromkeys(k for r in rows for k in r)) or ['status']
    with path.open('w',encoding='utf-8-sig',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=keys);writer.writeheader();writer.writerows(rows)


def read_json(path):return json.loads(Path(path).read_text(encoding='utf-8'))


def prepare(source,output,model,base_url,seeds,prices,arms):
    source=Path(source).resolve();output=Path(output).resolve();output.mkdir(parents=True,exist_ok=True)
    assert seeds and len(set(seeds))==len(seeds)
    assert prices and all(p>0 for p in prices) and len(set(prices))==len(prices)
    assert arms and set(arms)<=set(ARMS) and len(set(arms))==len(arms)
    metadata=read_json(source/'data/runs/prompt_probe_r1_current/metadata.json')
    params=metadata['parameters']
    assert params['INFINITE_ROUNDS'] is True
    assert params['DIVIDEND_PARAMS']['dividend_frequency']==1
    assert not params['AGENT_PARAMS']['leverage_params']['enabled']
    assert not params['AGENT_PARAMS']['allow_short_selling']
    params=deepcopy(params);params['AGENT_PARAMS']['leverage_params']['cash_lending_pool']=0.0
    params['NUM_ROUNDS']=15;params['MODEL_OPEN_AI']=model;params['LLM_MAX_CONCURRENCY']=8
    prompts={}
    for arm in arms:
        path=source/f'data/runs/prompt_probe_r1_{arm}/data/rendered_prompts.jsonl'
        rendered=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
        mm=[r['system_prompt'] for r in rendered if str(r['agent_id']) in MAKERS]
        assert len(mm)==2 and len(set(mm))==1
        prompts[arm]=mm[0]
    code_paths=['src/base_sim.py','src/order_lifecycle_audit.py','src/step4_lifecycle_audit.py',
                'src/market/orders/order_state_manager.py','src/market/orders/order_service_factory.py',
                'scripts/run_step4_longitudinal.py']
    hashes={p:hashlib.sha256((source/p).read_text(encoding='utf-8').encode()).hexdigest() for p in code_paths}
    paths={}
    dm=params['DIVIDEND_PARAMS']
    for seed in seeds:
        rng=random.Random(seed)
        paths[str(seed)]=[{'round_index':i,'market_round':i+1,
                           'dividend_per_share':dm['base_dividend']+(dm['dividend_variation'] if rng.random()<dm['dividend_probability'] else -dm['dividend_variation'])}
                          for i in range(15)]
    core={'protocol_version':VERSION,'source_commit':'074faff3c8f6255c917d2a2593a5b476c947ca84',
          'source_hashes':hashes,'model':model,'api_base_url':base_url.rstrip('/') or 'https://api.openai.com/v1',
          'temperature':0.0,'max_completion_tokens':2000,'seeds':seeds,'initial_prices':prices,
          'arms':arms,'maker_system_prompts':prompts,'parameters':params,'rounds':15,'agents':AGENTS,
          'maker_ids':sorted(MAKERS),'dividend_paths':paths,
          'dividend_path_hashes':{k:digest(v) for k,v in paths.items()},
          'processing_policy':'Agent 0-7; serial requests; fixed round-agent-order priorities; market orders then limit orders; no shuffle',
          'longitudinal_policy':'All 8 agents receive updated market state and generate responses every round; nonmaker responses are not frozen',
          'infinite_horizon':True,'redemption_value':None,'news_enabled':False}
    experiment_id=digest(core)[:20]
    trials=[]
    for seed in seeds:
        for price in prices:
            for arm in arms:
                key=f"seed{seed}_p{price:g}_{arm}"
                trials.append({'trial_key':key,'seed':seed,'initial_price':price,'arm':arm,
                               'dividend_path_sha256':core['dividend_path_hashes'][str(seed)]})
    random.Random(20261105).shuffle(trials)
    plan={**core,'experiment_id':experiment_id,'created_at_utc':datetime.now(timezone.utc).isoformat(),
          'trials':trials,'planned_model_requests':len(trials)*15*8,
          'request_count_formula':'number_of_trials * 15 rounds * 8 agents; no automatic retries'}
    save_json(output/'experiment_plan.json',plan)
    write_csv(output/'planned_trials.csv',trials)
    write_csv(output/'shared_dividend_paths.csv',[{'seed':seed,**row} for seed,rows in paths.items() for row in rows])
    for trial in trials:
        assert digest(paths[str(trial['seed'])])==trial['dividend_path_sha256']
    print(json.dumps({'status':'PLAN_READY','experiment_id':experiment_id,'trials':len(trials),
                      'rounds_per_trial':15,'planned_model_requests':plan['planned_model_requests'],
                      'new_model_requests_this_command':0}),flush=True)


class ExperimentAbort(BaseException):pass


class SerialExecutor:
    '''Execute the original collector's submissions serially, preserving its business logic.'''
    def __init__(self,*args,**kwargs):pass
    def __enter__(self):return self
    def __exit__(self,*args):return False
    def submit(self,fn,*args,**kwargs):
        result=fn(*args,**kwargs)
        future=Future();future.set_result(result);return future


class ResponseProvider:
    def __init__(self,plan,trial,output,cache,mode,max_new_requests):
        self.plan=plan;self.trial=trial;self.output=Path(output);self.cache=Path(cache)
        self.mode=mode;self.max_new=max_new_requests;self.new_calls=0;self.cached_calls=0
        self.records=[];self.sim=None;self.client=None
        self.cache.mkdir(parents=True,exist_ok=True)

    def event(self,entry):
        with (self.output/'model_requests.jsonl').open('a',encoding='utf-8') as f:
            f.write(json.dumps(entry,ensure_ascii=False)+'\n')

    def get(self,request):
        from agents.agents_api import TradeDecision
        from agents.LLMs.services.llm_services import LLMResponse
        from agents.LLMs.services.schema_features import FeatureRegistry
        core={'experiment_id':self.plan['experiment_id'],'trial_key':self.trial['trial_key'],
              'api_base_url':self.plan['api_base_url'],'model':self.plan['model'],
              'round_index':int(request.round_number),'agent_id':str(request.agent_id),
              'temperature':self.plan['temperature'],'seed':self.trial['seed'],
              'max_completion_tokens':self.plan['max_completion_tokens'],
              'system_prompt':request.system_prompt,'user_prompt':request.user_prompt}
        key=digest(core);path=self.cache/(key+'.json')
        raw=None;record=None
        if self.mode=='api_experiment' and path.exists():
            saved=read_json(path)
            if saved.get('status')=='complete' and saved.get('request_sha256')==key:
                raw=saved['parsed_response'];self.cached_calls+=1
                record={**saved,'response_source':'cached_actual_api_response'}
        if raw is None and self.mode=='scripted_smoke':
            aid=str(request.agent_id);price=float(self.sim.context.current_price)
            orders=[]
            if aid in MAKERS:
                orders=[{'decision':'Buy','quantity':100,'order_type':'limit','price_limit':round(price*0.99,2)},
                        {'decision':'Sell','quantity':100,'order_type':'limit','price_limit':round(price*1.01,2)}]
            elif aid=='0':orders=[{'decision':'Buy','quantity':100,'order_type':'market','price_limit':None}]
            elif aid=='1':orders=[{'decision':'Sell','quantity':100,'order_type':'market','price_limit':None}]
            raw={'valuation_reasoning':'SCRIPTED SMOKE FIXTURE; not model evidence','valuation':28,
                 'price_prediction_reasoning':'SCRIPTED SMOKE FIXTURE','price_prediction_t':price,
                 'price_prediction_t1':price,'price_prediction_t2':price,'reasoning':'SCRIPTED SMOKE FIXTURE',
                 'orders':orders,'replace_decision':'Replace'}
            record={'status':'complete','request_sha256':key,'request':core,'parsed_response':raw,
                    'response_source':'scripted_smoke_fixture','usage':None}
        if raw is None:
            if self.new_calls>=self.max_new:raise ExperimentAbort('New-request limit reached; completed response caches are retained.')
            if self.client is None:
                from openai import OpenAI
                api_key=os.environ.get('AHE_STEP4_API_KEY')
                if not api_key:raise ExperimentAbort('API mode requires an explicitly supplied key.')
                self.client=OpenAI(api_key=api_key,base_url=self.plan['api_base_url'],timeout=60.0,max_retries=0)
            self.new_calls+=1
            started={'event':'started','request_sha256':key,'round_index':request.round_number,
                     'agent_id':str(request.agent_id),'trial_key':self.trial['trial_key']}
            self.event(started)
            try:
                schema=FeatureRegistry.get_schema_for_features(set(),is_multi_stock=False)
                parse=getattr(self.client.chat.completions,'parse',None) or self.client.beta.chat.completions.parse
                completion=parse(model=core['model'],messages=[{'role':'system','content':request.system_prompt},
                                  {'role':'user','content':request.user_prompt}],response_format=schema,
                                  temperature=core['temperature'],seed=core['seed'],max_completion_tokens=core['max_completion_tokens'])
                message=completion.choices[0].message
                if getattr(message,'refusal',None) or message.parsed is None:raise ValueError('Unparsed/refused response')
                raw=message.parsed.model_dump(mode='json')
                TradeDecision.model_validate(raw)
                record={'status':'complete','request_sha256':key,'request':core,'parsed_response':raw,
                        'response_source':'new_actual_api_response','completion_id':getattr(completion,'id',None),
                        'returned_model':getattr(completion,'model',None),'system_fingerprint':getattr(completion,'system_fingerprint',None),
                        'usage':completion.usage.model_dump() if getattr(completion,'usage',None) else None}
                save_json(path,record)
                self.event({**started,'event':'complete'})
            except Exception as error:
                self.event({**started,'event':'failed','error_type':type(error).__name__,
                            'http_status':getattr(error,'status_code',None)})
                raise ExperimentAbort(f'API request stopped: {type(error).__name__}; no automatic retry or model replacement.') from None
        try:
            decision=TradeDecision.model_validate(raw)
        except Exception as error:raise ExperimentAbort('Decision validation failed: '+type(error).__name__) from None
        self.records.append(record)
        with (self.output/'responses_used.jsonl').open('a',encoding='utf-8') as f:
            f.write(json.dumps(record,ensure_ascii=False)+'\n')
        body=decision.model_dump(mode='json');body['orders']=decision.orders;body['agent_id']=str(request.agent_id)
        return LLMResponse(decision=body,raw_response=json.dumps(raw,ensure_ascii=False))


def run_trial(plan,trial,source,root,cache,mode,max_new_requests):
    from base_sim import BaseSimulation
    from agents.LLMs.services.llm_services import LLMService
    from agents.agent_manager.services import agent_decision_service as collector_module
    from market.state.services.dividend_service import DividendRealization
    from step4_lifecycle_audit import LifecycleAudit
    from services.logging_service import LoggingService
    from scenarios.base import FundamentalInfoMode
    for relative,expected in plan['source_hashes'].items():
        if hashlib.sha256((source/relative).read_text(encoding='utf-8').encode()).hexdigest()!=expected:
            raise ExperimentAbort('Source changed since planning: '+relative)
    trial_root=Path(root)/'runs'/trial['trial_key'];trial_root.mkdir(parents=True,exist_ok=True)
    output=trial_root/('attempt_'+uuid.uuid4().hex[:10]);output.mkdir()
    provider=ResponseProvider(plan,trial,output,cache,mode,max_new_requests)
    original_init=LLMService.__init__;original_get=LLMService.get_decision
    original_fallback=LLMService.get_fallback_decision;original_executor=collector_module.ThreadPoolExecutor
    LLMService.__init__=lambda self:None
    LLMService.get_decision=lambda self,request:provider.get(request)
    def no_fallback(*args,**kwargs):raise ExperimentAbort('The agent entered a fallback path; experiment stopped.')
    LLMService.get_fallback_decision=no_fallback;collector_module.ThreadPoolExecutor=SerialExecutor
    params=deepcopy(plan['parameters']);params['AGENT_PARAMS']['max_order_size']=1000
    random.seed(trial['seed'])
    import numpy as np
    np.random.seed(trial['seed'])
    sim=None;audit=None
    try:
        sim=BaseSimulation(num_rounds=15,initial_price=trial['initial_price'],fundamental_price=params['FUNDAMENTAL_PRICE'],
            redemption_value=None,transaction_cost=0.0,lendable_shares=0,agent_params=params['AGENT_PARAMS'],
            fundamental_info_mode=FundamentalInfoMode.PROCESS_ONLY,model_open_ai=plan['model'],
            llm_temperature=0.0,llm_seed=trial['seed'],llm_max_concurrency=8,
            system_prompt_overrides={'market_maker':plan['maker_system_prompts'][trial['arm']]},
            dividend_params=params['DIVIDEND_PARAMS'],interest_params=params['INTEREST_MODEL'],
            infinite_rounds=True,sim_type=f"step4/{plan['experiment_id']}/{mode}/{trial['trial_key']}/{output.name}",news_enabled=False)
        provider.sim=sim
        # Detailed market logs stay in this attempt; no shared latest directory is analyzed.
        save_json(output/'trial_metadata.json',{'mode':mode,'trial':trial,'experiment_id':plan['experiment_id'],
                                               'simulator_log_directory':str(sim.run_dir.resolve())})
        for logger in LoggingService._loggers.values():
            for handler in logger.handlers:
                if isinstance(handler,logging.StreamHandler) and not isinstance(handler,logging.FileHandler):
                    handler.setLevel(logging.ERROR)
        audit=LifecycleAudit(sim.context,sim.agent_repository,sim.order_repository,output/'audit',
                             run_id=plan['experiment_id']+':'+trial['trial_key']+':'+output.name)
        audit.attach(sim.decision_service,sim.order_state_manager,sim.trade_execution_service,sim.matching_engine,sim.data_recorder)
        initial_resources=audit.all_resources()
        assert [str(a) for a in sim.agent_repository.get_all_agent_ids()]==AGENTS
        native_agent_ids=list(sim.agent_repository.get_all_agent_ids())
        sim.agent_repository.get_shuffled_agent_ids=lambda:list(native_agent_ids)
        active_round={'index':0};counts=Counter()
        original_create=sim.decision_service.create_order
        def fixed_create(details,agent_id):
            order=original_create(details,agent_id);r=active_round['index'];aid=str(agent_id)
            index=counts[(r,aid)];counts[(r,aid)]+=1
            assert index<1000
            order.step4_priority=(r,int(aid),index)
            order.timestamp=1_000_000_000.0+r*1_000_000+int(aid)*1000+index
            audit.emit('CONTROLLED_PRIORITY_ASSIGNED',order,priority_key=list(order.step4_priority))
            return order
        sim.decision_service.create_order=fixed_create
        def fixed_split(orders):
            ordered=sorted(orders,key=lambda o:o.step4_priority)
            return [o for o in ordered if o.order_type=='market'],[o for o in ordered if o.order_type=='limit']
        sim.matching_engine.order_processing_service.split_orders_by_type=fixed_split
        path=plan['dividend_paths'][str(trial['seed'])];realizations=[];snapshots=[];round_metrics=[]
        def fixed_dividend(systematic_shock=0.0,style_shock=0.0):
            r=active_round['index'];value=path[r]['dividend_per_share']
            assert systematic_shock==0 and style_shock==0 and all(x['round_index']!=r for x in realizations)
            realizations.append({'trial_key':trial['trial_key'],'seed':trial['seed'],**path[r]})
            return DividendRealization(total_dividend=value,base_component=params['DIVIDEND_PARAMS']['base_dividend'],
                                       idiosyncratic_component=value-params['DIVIDEND_PARAMS']['base_dividend'])
        sim.dividend_service.calculator.calculate_dividend=fixed_dividend
        def snapshot(phase,r,pending=None):
            book=sim.order_book;bid=book.get_best_bid();ask=book.get_best_ask()
            snapshots.append({'trial_key':trial['trial_key'],'round_index':r,'market_round':r+1,'phase':phase,
                              'recorded_price':sim.context.current_price,'best_bid':bid,'best_ask':ask,
                              'two_sided_midpoint':(bid+ask)/2 if bid is not None and ask is not None else None,
                              'spread':ask-bid if bid is not None and ask is not None else None,
                              'resting_buy_orders':len(book.buy_orders),'resting_sell_orders':len(book.sell_orders),
                              'accepted_new_orders_waiting_for_matching':pending})
        original_update=sim._phase_update_market
        def update(r):
            state=original_update(r);snapshot('pre_decision',r);return state
        sim._phase_update_market=update
        original_match=sim._phase_match_orders
        def match(new_orders,r):
            snapshot('after_validation_pre_matching',r,len(new_orders))
            result=original_match(new_orders,r);snapshot('post_matching',r);return result
        sim._phase_match_orders=match
        for r in range(15):
            active_round['index']=r;audit.experiment_round_index=r
            start_trades=len(audit.trades);start_decisions=len(audit.decisions)
            before_price=float(sim.context.current_price)
            sim.execute_round(r)
            rows=audit.validate();snapshot('round_end_after_payments',r);audit.snapshot('round_end_after_payments')
            assert len(audit.decisions)-start_decisions==8 and len(provider.records)==(r+1)*8
            assert len(realizations)==r+1 and len(sim.dividend_service.dividend_history)==r+1
            assert abs(sim.dividend_service.dividend_history[-1].total_dividend-path[r]['dividend_per_share'])<1e-12
            by_id={row['order_id']:row for row in rows}
            decision_slice=audit.decisions[start_decisions:]
            maker_decisions=[d for d in decision_slice if d['agent_id'] in MAKERS]
            proposed=sum({o['decision'].lower() for o in d['response']['orders']}=={'buy','sell'} for d in maker_decisions)
            accepted_both=sum({o['side'] for o in rows if o['decision_id']==d['decision_id'] and o['accepted']}=={'buy','sell'} for d in maker_decisions)
            live=sum({entry.order.side for entry in sim.order_book.buy_orders+sim.order_book.sell_orders
                      if str(entry.order.agent_id)==aid and entry.order.remaining_quantity>0}=={'buy','sell'} for aid in MAKERS)
            ts=audit.trades[start_trades:];volume=sum(float(t['quantity']) for t in ts)
            weighted=sum(float(t['quantity'])*float(t['price']) for t in ts)/volume if volume else None
            for row in rows:
                if row['accepted']:assert row['accepted_quantity']<=1000
            post=next(s for s in reversed(snapshots) if s['phase']=='post_matching')
            round_metrics.append({'trial_key':trial['trial_key'],'arm':trial['arm'],'seed':trial['seed'],
                'initial_price':trial['initial_price'],'round_index':r,'market_round':r+1,
                'maker_decisions':2,'maker_proposed_two_sided':proposed,'maker_accepted_two_sided':accepted_both,
                'makers_with_two_sided_resting_quotes_post_matching':live,
                'trade_count':len(ts),'executed_shares':volume,'trades_using_maker_bids':sum(t['buyer_id'] in MAKERS for t in ts),
                'recorded_price':float(sim.context.current_price),'recorded_price_change':float(sim.context.current_price)-before_price,
                'execution_vwap_this_round':weighted,'last_execution_this_round':float(ts[-1]['price']) if ts else None,
                'post_matching_best_bid':post['best_bid'],'post_matching_best_ask':post['best_ask'],
                'post_matching_two_sided_midpoint':post['two_sided_midpoint'],'actual_dividend_per_share':realizations[-1]['dividend_per_share']})
            write_csv(output/'round_metrics.csv',round_metrics);write_csv(output/'quote_snapshots.csv',snapshots)
            write_csv(output/'dividend_realizations.csv',realizations)
            save_json(output/'progress.json',{'status':'RUNNING','completed_rounds':r+1,'new_requests_this_attempt':provider.new_calls})
            print(f"{trial['trial_key']} round {r+1}/15: checks PASS; trades={len(ts)}; new API requests={provider.new_calls}",flush=True)
        lifecycle=audit.export(sim.data_recorder);sim.data_recorder.save_simulation_data()
        assert realizations==[{'trial_key':trial['trial_key'],'seed':trial['seed'],**x} for x in path]
        assert len(provider.records)==120 and len(snapshots)==60
        accepted=[o for o in lifecycle if o['accepted']]
        volume=sum(r['executed_shares'] for r in round_metrics)
        executed=[t for t in audit.trades]
        true_errors=sum(float(t['quantity'])*abs(float(t['price'])-28.0) for t in executed)/volume if volume else None
        last_trade=float(executed[-1]['price']) if executed else None
        accepted_qty=sum(o['accepted_quantity'] for o in accepted)
        normalized={'round_metrics':[{k:v for k,v in row.items() if k!='trial_key'} for row in round_metrics],
                    'resources':audit.all_resources(),'orders':[{'priority':list(sim.order_repository.get_order(o['order_id']).step4_priority),
                     **{k:o[k] for k in ['agent_id','side','requested_quantity','accepted','accepted_quantity','executed_quantity',
                                         'remaining_quantity_raw','post_acceptance_removed_quantity','final_state']}} for o in lifecycle]}
        summary={'trial_key':trial['trial_key'],'arm':trial['arm'],'seed':trial['seed'],'initial_price':trial['initial_price'],
            'mode':mode,'rounds_completed':15,'agent_decisions':120,'maker_decisions':30,
            'maker_proposed_two_sided':sum(r['maker_proposed_two_sided'] for r in round_metrics),
            'maker_accepted_two_sided':sum(r['maker_accepted_two_sided'] for r in round_metrics),
            'trade_count':len(executed),'executed_shares':volume,'trades_using_maker_bids':sum(t['buyer_id'] in MAKERS for t in executed),
            'accepted_orders':len(accepted),'rejected_orders':len(lifecycle)-len(accepted),
            'oversized_rejections':sum(str(o['validation_reason']).startswith('MAX_ORDER_SIZE_EXCEEDED') for o in lifecycle),
            'fully_filled_orders':sum(o['execution_status']=='fully_filled' for o in lifecycle),
            'partially_filled_orders':sum(o['execution_status']=='partially_filled' for o in lifecycle),
            'accepted_unfilled_orders':sum(o['execution_status']=='unfilled' for o in lifecycle),
            'accepted_quantity_fill_rate':2*volume/accepted_qty if accepted_qty else None,
            'final_recorded_price':round_metrics[-1]['recorded_price'],'last_actual_trade_price':last_trade,
            'recorded_price_mean_absolute_error':sum(abs(r['recorded_price']-28.0) for r in round_metrics)/15,
            'volume_weighted_actual_trade_price_absolute_error':true_errors,
            'zero_trade_rounds':sum(r['trade_count']==0 for r in round_metrics),
            'absolute_recorded_price_change_in_zero_trade_rounds':sum(abs(r['recorded_price_change']) for r in round_metrics if not r['trade_count']),
            'dividend_path_sha256':digest(path),'initial_resources_sha256':digest(initial_resources),
            'new_model_requests_this_attempt':provider.new_calls,'cached_actual_responses_used':provider.cached_calls,
            'used_model_response_count':len(provider.records) if mode=='api_experiment' else 0,
            'scripted_fixture_response_count':len(provider.records) if mode=='scripted_smoke' else 0,
            'source_checks':'PASS','dividend_path_checks':'PASS','order_trade_resource_checks':'PASS',
            'outcome_sha256':digest(normalized),'attempt_directory':str(output.resolve())}
        save_json(output/'run_summary.json',summary);save_json(output/'normalized_outcome.json',normalized)
        save_json(output/'progress.json',{'status':'COMPLETE','completed_rounds':15})
        save_json(trial_root/'completed.json',summary)
        return summary
    except BaseException as error:
        save_json(output/'progress.json',{'status':'FAILED','error_type':type(error).__name__,
                                        'completed_response_records':len(provider.records),'new_requests_this_attempt':provider.new_calls})
        if audit is not None:
            try:audit.export(sim.data_recorder)
            except Exception:pass
        raise
    finally:
        LoggingService.reset()
        LLMService.__init__=original_init;LLMService.get_decision=original_get
        LLMService.get_fallback_decision=original_fallback;collector_module.ThreadPoolExecutor=original_executor


def run_suite(plan,source,output,cache,mode,max_new):
    output=Path(output).resolve();cache=Path(cache).resolve()
    output.mkdir(parents=True,exist_ok=True);os.chdir(output)
    if mode=='scripted_smoke':
        def deny(*args,**kwargs):raise ExperimentAbort('Smoke worker attempted a network connection')
        socket.socket.connect=deny;socket.socket.connect_ex=deny;socket.create_connection=deny
        # A pair plus an identical rerun verifies the controls, not LLM behavior.
        trials=[deepcopy(plan['trials'][0])]
        second=deepcopy(trials[0]);second['arm']=next((a for a in plan['arms'] if a!=second['arm']),second['arm'])
        second['trial_key']=second['trial_key']+'_smoke_other_arm'
        trials.append(second)
    else:trials=plan['trials']
    summaries=[];spent=0
    for trial in trials:
        completed=output/'runs'/trial['trial_key']/'completed.json'
        if mode=='api_experiment' and completed.exists():
            prior=read_json(completed)
            if Path(prior['attempt_directory']).joinpath('run_summary.json').is_file():summaries.append(prior);continue
        summary=run_trial(plan,trial,source,output,cache,mode,max_new-spent)
        summaries.append(summary);spent+=summary['new_model_requests_this_attempt']
        write_csv(output/'trial_summary.csv',summaries)
    if mode=='scripted_smoke':
        again=run_trial(plan,trials[0],source,output/'reproducibility_check',cache,mode,0)
        assert summaries[0]['outcome_sha256']==again['outcome_sha256']
        assert all(s['new_model_requests_this_attempt']==0 and s['agent_decisions']==120 for s in summaries)
        assert len({s['dividend_path_sha256'] for s in summaries})==1
        assert len({s['initial_resources_sha256'] for s in summaries})==1
    else:
        assert len(summaries)==len(plan['trials'])
        for seed in plan['seeds']:
            assert {s['dividend_path_sha256'] for s in summaries if s['seed']==seed}=={plan['dividend_path_hashes'][str(seed)]}
        for price in plan['initial_prices']:
            assert len({s['initial_resources_sha256'] for s in summaries if s['initial_price']==price})==1
    write_csv(output/'trial_summary.csv',summaries)
    checks={'mode':mode,'status':'PASS','completed_trials':len(summaries),
            'rounds_per_trial':15,'new_model_requests_this_execution':spent,
            'dividend_paths_match_plan':True,'order_trade_resource_linkage':True,
            'smoke_repeat_reproducibility':True if mode=='scripted_smoke' else None,
            'research_results_available':mode=='api_experiment',
            'scope':'Scripted plumbing checks only' if mode=='scripted_smoke' else 'Paired dividend paths; all agents respond endogenously; descriptive small-sample comparison'}
    save_json(output/'suite_checks.json',checks)
    print('SUITE '+json.dumps(checks),flush=True)


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('command',choices=['prepare','smoke','run'])
    parser.add_argument('--source',type=Path,required=True)
    parser.add_argument('--output',type=Path,required=True)
    parser.add_argument('--plan',type=Path)
    parser.add_argument('--cache-dir',type=Path)
    parser.add_argument('--model',default='gpt-4o-2024-11-20')
    parser.add_argument('--base-url',default='')
    parser.add_argument('--seeds',default='[42,43,44]')
    parser.add_argument('--prices',default='[56.0]')
    parser.add_argument('--arms',default=json.dumps(ARMS))
    parser.add_argument('--max-new-requests',type=int,default=1080)
    parser.add_argument('--allow-api',action='store_true')
    args=parser.parse_args();source=args.source.resolve()
    if args.command=='prepare':
        prepare(source,args.output,args.model,args.base_url,json.loads(args.seeds),json.loads(args.prices),json.loads(args.arms));return
    if args.command=='run' and not args.allow_api:raise RuntimeError('API mode requires explicit --allow-api')
    plan=read_json(args.plan)
    sys.path.insert(0,str(source/'src'))
    mode='scripted_smoke' if args.command=='smoke' else 'api_experiment'
    try:run_suite(plan,source,args.output,args.cache_dir or args.output/'smoke_only_cache',mode,args.max_new_requests)
    except ExperimentAbort as error:
        print(str(error),file=sys.stderr,flush=True);raise SystemExit(2)


if __name__=='__main__':main()
""",
'src/step4_lifecycle_audit.py': r"""'''Observe real decision, order, resource and trade services without changing their rules.

Single-stock audit support for the AI Herding research branch. Attach before
collecting decisions. Persisted quantities distinguish acceptance-time resizing,
post-acceptance resizing, actual executions, and cancelled remainders.
'''
from __future__ import annotations

import csv
import json
import threading
import uuid
from collections import defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone
from enum import Enum
from functools import wraps
from pathlib import Path

from market.orders.order import Order, OrderState


def json_default(value):
    if isinstance(value, Enum):
        return value.value
    if isinstance(value, (datetime, Path)):
        return str(value)
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='json')
    if hasattr(value, 'item'):
        return value.item()
    raise TypeError(f'Unsupported audit value: {type(value).__name__}')


def clean(value):
    return json.loads(json.dumps(value, default=json_default, ensure_ascii=False))


def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with path.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=keys or ['run_id'])
        writer.writeheader()
        for row in rows:
            writer.writerow({key: json.dumps(value, ensure_ascii=False, default=json_default)
                             if isinstance(value, (dict, list)) else value
                             for key, value in row.items()})


class ObservedOrder(Order):
    '''Preserve Order behavior and observe assignments on attached instances only.'''
    _observed_fields = {
        'quantity', 'remaining_quantity', 'filled_quantity', 'state', 'price',
        'current_cash_commitment', 'current_share_commitment',
    }

    def __setattr__(self, name, value):
        observer = self.__dict__.get('_lifecycle_observer')
        old = self.__dict__.get(name)
        existed = name in self.__dict__
        object.__setattr__(self, name, value)
        if observer is not None and existed and name in self._observed_fields and old != value:
            observer.attribute_changed(self, name, old, value)


class LifecycleAudit:
    def __init__(self, context, agent_repository, order_repository, output_dir, run_id=None):
        self.context = context
        # The multi-round runner supplies its explicit decision-round index.
        self.experiment_round_index = int(context.round_number)
        self.agents = agent_repository
        self.orders = order_repository
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.run_id = run_id or 'lifecycle_' + uuid.uuid4().hex
        self.lock = threading.RLock()
        self.local = threading.local()
        self.event_seq = 0
        self.id_counter = defaultdict(int)
        self.events = []
        self.decisions = []
        self.proposals = []
        self.meta = {}
        self.pending_decisions = {}
        self.trades = []
        self.rounds = []
        self.order_snapshots = []
        self.agent_snapshots = []
        self.bound_methods = []
        self.event_path = self.output_dir / 'order_events.jsonl'
        if self.event_path.exists():
            raise FileExistsError(f'Choose a fresh audit output directory: {self.output_dir}')

    def new_id(self, kind):
        with self.lock:
            self.id_counter[kind] += 1
            return f'{self.run_id}:{kind}:{self.id_counter[kind]:05d}'

    @contextmanager
    def operation(self, **updates):
        old = getattr(self.local, 'operation', {})
        self.local.operation = {**old, **updates}
        try:
            yield
        finally:
            self.local.operation = old

    def resource(self, agent_id, stock_id='DEFAULT_STOCK'):
        agent = self.agents.get_agent(agent_id)
        return {
            'available_cash': float(agent.available_cash),
            'committed_cash': float(agent.committed_cash),
            'dividend_cash': float(agent.dividend_cash),
            'total_cash': float(agent.total_cash),
            'available_shares': int(agent.positions.get(stock_id, 0)),
            'committed_shares': int(agent.committed_positions.get(stock_id, 0)),
            'borrowed_shares': int(agent.borrowed_positions.get(stock_id, 0)),
        }

    def all_resources(self):
        return {str(aid): self.resource(aid) for aid in self.agents.get_all_agent_ids()}

    def order_view(self, order):
        return {
            'order_id': str(order.order_id), 'agent_id': str(order.agent_id),
            'stock_id': order.stock_id, 'side': order.side, 'order_type': order.order_type,
            'quantity': order.quantity, 'remaining_quantity': order.remaining_quantity,
            'filled_quantity': order.filled_quantity, 'state': order.state.value,
            'price': order.price, 'cash_commitment': order.current_cash_commitment,
            'share_commitment': order.current_share_commitment,
        }

    def emit(self, event_type, order=None, agent_id=None, before=None, after=None, **details):
        op = dict(getattr(self.local, 'operation', {}))
        oid = str(order.order_id) if order is not None else op.get('order_id', '')
        meta = self.meta.get(oid, {})
        if agent_id is None and order is not None:
            agent_id = order.agent_id
        row = {
            'run_id': self.run_id, 'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            'round_index': int(self.experiment_round_index),
            'market_round': int(self.experiment_round_index) + 1,
            'event_type': event_type, 'agent_id': '' if agent_id is None else str(agent_id),
            'order_id': oid, 'decision_id': meta.get('decision_id', op.get('decision_id', '')),
            'proposal_id': meta.get('proposal_id', op.get('proposal_id', '')),
            'trigger_decision_id': op.get('trigger_decision_id', ''),
            'trade_id': op.get('trade_id', ''), 'activity': op.get('activity', ''),
            'buyer_order_id': op.get('buyer_order_id', ''),
            'seller_order_id': op.get('seller_order_id', ''),
            'resource_before': before, 'resource_after': after,
            'order': self.order_view(order) if order is not None else None,
            'details': {**{key: value for key, value in op.items()
                          if key in ('transition_notes', 'conversion_reason')}, **details},
        }
        with self.lock:
            self.event_seq += 1
            row['event_seq'] = self.event_seq
            row = clean(row)
            self.events.append(row)
            with self.event_path.open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(row, ensure_ascii=False) + '\n')
        return row

    def attribute_changed(self, order, field, old, new):
        meta = self.meta[str(order.order_id)]
        op = getattr(self.local, 'operation', {})
        if field == 'remaining_quantity' and meta['accepted_quantity'] is not None and op.get('activity') != 'trade_execution':
            meta['post_acceptance_removed_quantity'] += float(old) - float(new)
        event_type = {
            'state': 'STATE_CHANGED', 'quantity': 'ORDER_QUANTITY_CHANGED',
            'remaining_quantity': 'ORDER_REMAINING_CHANGED',
            'filled_quantity': 'ORDER_FILLED_COUNTER_CHANGED', 'price': 'ORDER_PRICE_CHANGED',
            'current_cash_commitment': 'ORDER_CASH_COMMITMENT_CHANGED',
            'current_share_commitment': 'ORDER_SHARE_COMMITMENT_CHANGED',
        }[field]
        self.emit(event_type, order, after=self.resource(order.agent_id, order.stock_id),
                  field=field, value_before=old, value_after=new,
                  is_execution=op.get('activity') == 'trade_execution')

    def bind(self, obj, name, wrapper):
        original = getattr(obj, name)
        self.bound_methods.append((obj, name, original))
        setattr(obj, name, wraps(original)(wrapper(original)))

    def current_decision(self, agent_id):
        return self.pending_decisions.get((int(self.experiment_round_index), str(agent_id)))

    def attach(self, decision_service, state_manager, trade_service, engine, recorder=None):
        if self.orders.orders:
            raise ValueError('Attach the audit before creating orders; old missing events cannot be recreated')

        def wrap_decision(original):
            def call(agent_id, market_state, history, round_number):
                before = self.resource(agent_id)
                response = original(agent_id, market_state, history, round_number)
                decision = {
                    'run_id': self.run_id, 'decision_id': self.new_id('decision'),
                    'agent_id': str(agent_id), 'round_index': int(round_number),
                    'market_round': int(round_number) + 1,
                    'replace_decision': response['replace_decision'],
                    'response': clean(response), 'cancelled_order_ids': [],
                    'resource_at_decision': before,
                }
                with self.lock:
                    self.decisions.append(decision)
                    self.pending_decisions[(int(round_number), str(agent_id))] = decision
                with self.operation(activity='decision_collection', decision_id=decision['decision_id']):
                    self.emit('DECISION_RECORDED', agent_id=agent_id, before=before,
                              replace_decision=decision['replace_decision'], response=decision['response'])
                return response
            return call
        self.bind(self.agents, 'get_agent_decision', wrap_decision)

        def wrap_create(original):
            def call(order_details, agent_id):
                decision = self.current_decision(agent_id)
                if decision is None:
                    raise ValueError('No recorded decision for this order creation')
                raw = clean(order_details)
                proposal = {
                    'run_id': self.run_id, 'decision_id': decision['decision_id'],
                    'proposal_id': self.new_id('proposal'), 'agent_id': str(agent_id),
                    'round_index': int(self.experiment_round_index),
                    'requested_quantity': raw['quantity'], 'order_id': '',
                    'replace_decision': decision['replace_decision'],
                    'replaces_order_ids': list(decision['cancelled_order_ids'])
                        if decision['replace_decision'] == 'Replace' else [],
                    'request': raw,
                }
                self.proposals.append(proposal)
                with self.operation(activity='order_creation', decision_id=decision['decision_id'],
                                    proposal_id=proposal['proposal_id'], proposal=proposal):
                    self.emit('ORDER_PROPOSED', agent_id=agent_id, before=self.resource(agent_id), request=raw)
                    try:
                        return original(order_details, agent_id)
                    except Exception as error:
                        proposal['creation_error'] = str(error)
                        self.emit('ORDER_CREATION_REJECTED', agent_id=agent_id, reason=str(error))
                        raise
            return call
        self.bind(decision_service, 'create_order', wrap_create)

        def wrap_register(original):
            def call(order):
                proposal = getattr(self.local, 'operation', {}).get('proposal')
                oid = str(order.order_id)
                if proposal is None:
                    # Broker-created orders are explicitly separate from agent decisions.
                    proposal = {'decision_id': '', 'proposal_id': self.new_id('engine_proposal'),
                                'requested_quantity': order.quantity, 'replaces_order_ids': []}
                proposal['order_id'] = oid
                self.meta[oid] = {
                    'decision_id': proposal['decision_id'], 'proposal_id': proposal['proposal_id'],
                    'requested_quantity': proposal['requested_quantity'], 'accepted_quantity': None,
                    'post_acceptance_removed_quantity': 0.0, 'accepted': False,
                    'validation_reason': '', 'cancellation_reason': '',
                    'replaces_order_ids': list(proposal['replaces_order_ids']),
                }
                order.__class__ = ObservedOrder
                object.__setattr__(order, '_lifecycle_observer', self)
                object.__setattr__(order, 'decision_id', proposal['decision_id'])
                object.__setattr__(order, 'proposal_id', proposal['proposal_id'])
                result = original(order)
                self.emit('ORDER_REGISTERED', order, after=self.resource(order.agent_id, order.stock_id),
                          replaces_order_ids=proposal['replaces_order_ids'])
                return result
            return call
        self.bind(self.orders, 'create_order', wrap_register)

        def wrap_transition(original):
            def call(order_id, new_state, *args, **kwargs):
                notes = kwargs.get('notes', args[2] if len(args) > 2 else '')
                with self.operation(order_id=str(order_id), transition_notes=notes or ''):
                    return original(order_id, new_state, *args, **kwargs)
            return call
        self.bind(self.orders, 'transition_state', wrap_transition)

        def wrap_validate(original):
            def call(order, current_price):
                before = self.resource(order.agent_id, order.stock_id)
                with self.operation(activity='initial_validation', order_id=str(order.order_id)):
                    self.emit('VALIDATION_STARTED', order, before=before)
                    valid, reason = original(order, current_price)
                    after = self.resource(order.agent_id, order.stock_id)
                    meta = self.meta[str(order.order_id)]
                    if meta['accepted_quantity'] is None:
                        meta['accepted'] = bool(valid)
                        meta['accepted_quantity'] = float(order.quantity) if valid else 0.0
                        meta['validation_reason'] = reason
                        meta['validation_resource_before'] = before
                        meta['validation_resource_after'] = after
                    self.emit('VALIDATION_ACCEPTED' if valid else 'VALIDATION_REJECTED',
                              order, before=before, after=after, reason=reason,
                              requested_quantity=meta['requested_quantity'],
                              accepted_quantity=meta['accepted_quantity'])
                    return valid, reason
            return call
        self.bind(state_manager, 'handle_new_order', wrap_validate)

        def resource_order(agent_id):
            op = getattr(self.local, 'operation', {})
            if op.get('trade_id'):
                oid = op['buyer_order_id'] if str(agent_id) == op.get('buyer_id') else op['seller_order_id']
            else:
                oid = op.get('order_id')
            return self.orders.orders.get(oid)

        def wrap_resource(method_name, label):
            def factory(original):
                def call(agent_id, *args, **kwargs):
                    stock = kwargs.get('stock_id', 'DEFAULT_STOCK')
                    before = self.resource(agent_id, stock)
                    order = resource_order(agent_id)
                    self.emit(label + '_REQUESTED', order, agent_id, before=before, request=clean(kwargs))
                    result = original(agent_id, *args, **kwargs)
                    success_event = 'RESOURCE_COMMITTED' if label == 'RESOURCE_COMMIT' else 'RESOURCE_RELEASED'
                    self.emit(success_event if result.success else label + '_REJECTED', order, agent_id,
                              before=before, after=self.resource(agent_id, stock),
                              success=bool(result.success), message=result.message,
                              partial_allocation=bool(getattr(result, 'partial_fill', False)),
                              requested_amount=getattr(result, 'requested_amount', None),
                              committed_amount=getattr(result, 'committed_amount', None))
                    return result
                return call
            self.bind(self.agents, method_name, factory)
        wrap_resource('commit_resources', 'RESOURCE_COMMIT')
        wrap_resource('release_resources', 'RESOURCE_RELEASE')

        def wrap_position(original):
            def call(agent_id, changes):
                order = resource_order(agent_id)
                stock = changes.stock_id
                before = self.resource(agent_id, stock)
                result = original(agent_id, changes)
                self.emit('POSITION_UPDATED', order, agent_id, before=before,
                          after=self.resource(agent_id, stock),
                          cash_change=changes.cash_change, share_change=changes.shares_change)
                return result
            return call
        self.bind(self.agents, 'update_agent_position_after_trade', wrap_position)

        def wrap_cancel(original):
            def call(order, message='Cancelled', skip_sync=False):
                decision = self.current_decision(order.agent_id)
                trigger = decision['decision_id'] if decision else ''
                before = self.resource(order.agent_id, order.stock_id)
                with self.operation(activity='cancellation', order_id=str(order.order_id),
                                    trigger_decision_id=trigger):
                    self.emit('CANCEL_STARTED', order, before=before, reason=message)
                    result = original(order, message, skip_sync=skip_sync)
                    self.meta[str(order.order_id)]['cancellation_reason'] = message
                    if decision is not None:
                        decision['cancelled_order_ids'].append(str(order.order_id))
                    self.emit('CANCEL_COMPLETED', order, before=before,
                              after=self.resource(order.agent_id, order.stock_id), reason=message)
                    return result
            return call
        self.bind(state_manager, 'handle_single_order_cancellation', wrap_cancel)

        def wrap_trade(original):
            def call(trade):
                tid = self.new_id('trade')
                object.__setattr__(trade, 'trade_id', tid)
                before = {str(trade.buyer_id): self.resource(trade.buyer_id, trade.stock_id),
                          str(trade.seller_id): self.resource(trade.seller_id, trade.stock_id)}
                with self.operation(activity='trade_execution', trade_id=tid,
                                    order_id=str(trade.buyer_order_id),
                                    buyer_id=str(trade.buyer_id), seller_id=str(trade.seller_id),
                                    buyer_order_id=str(trade.buyer_order_id),
                                    seller_order_id=str(trade.seller_order_id)):
                    self.emit('TRADE_EXECUTION_STARTED', before=before,
                              quantity=trade.quantity, price=trade.price)
                    success = original(trade)
                    after = {str(trade.buyer_id): self.resource(trade.buyer_id, trade.stock_id),
                             str(trade.seller_id): self.resource(trade.seller_id, trade.stock_id)}
                    event = self.emit('TRADE_EXECUTED' if success else 'TRADE_REJECTED',
                                      before=before, after=after,
                                      quantity=trade.quantity, price=trade.price)
                    if success:
                        self.trades.append({
                            'run_id': self.run_id, 'trade_id': tid, 'event_seq': event['event_seq'],
                            'round_index': int(self.experiment_round_index),
                            'market_round': int(self.experiment_round_index) + 1,
                            'buyer_id': str(trade.buyer_id), 'seller_id': str(trade.seller_id),
                            'buyer_order_id': str(trade.buyer_order_id),
                            'seller_order_id': str(trade.seller_order_id),
                            'buyer_decision_id': self.meta[str(trade.buyer_order_id)]['decision_id'],
                            'seller_decision_id': self.meta[str(trade.seller_order_id)]['decision_id'],
                            'quantity': trade.quantity, 'price': trade.price, 'stock_id': trade.stock_id,
                        })
                    return success
            return call
        self.bind(trade_service, 'execute_trade', wrap_trade)

        def quote():
            return {'best_bid': engine.order_book.get_best_bid(),
                    'best_ask': engine.order_book.get_best_ask(),
                    'midpoint': engine.order_book.get_midpoint()}

        def wrap_match(original):
            def call(new_orders, current_price, round_number):
                before = quote()
                trade_start_index = len(self.trades)
                with self.operation(activity='matching'):
                    self.emit('MATCHING_STARTED', quotes=before,
                              new_order_ids=[str(o.order_id) for o in new_orders])
                    result = original(new_orders, current_price, round_number)
                    actual_ids = {t['trade_id'] for t in self.trades[trade_start_index:]}
                    returned_ids = {getattr(t, 'trade_id', '') for t in result.trades}
                    if actual_ids != returned_ids:
                        raise AssertionError('Returned trades differ from successfully executed trades')
                    after = quote()
                    self.emit('MATCHING_COMPLETED', quotes=after, volume=result.volume,
                              trade_ids=sorted(returned_ids), recorded_price=result.price)
                    self.rounds.append({
                        'run_id': self.run_id, 'round_index': int(self.experiment_round_index),
                        'market_round': int(self.experiment_round_index) + 1, 'recorded_price': result.price,
                        'trade_count': len(result.trades), 'trade_volume': result.volume,
                        'last_execution_this_round': result.trades[-1].price if result.trades else None,
                        **{'pre_' + key: value for key, value in before.items()},
                        **{'post_' + key: value for key, value in after.items()},
                    })
                    return result
            return call
        self.bind(engine, 'match_orders', wrap_match)

        def wrap_processing(original):
            def call(orders):
                result = original(orders)
                self.emit('ORDER_PROCESSING_SEQUENCE',
                          market_order_ids=[str(o.order_id) for o in result[0]],
                          limit_order_ids=[str(o.order_id) for o in result[1]])
                return result
            return call
        self.bind(engine.order_processing_service, 'split_orders_by_type', wrap_processing)

        def wrap_conversion(original):
            def call(order, current_price):
                with self.operation(activity='conversion', order_id=str(order.order_id),
                                    conversion_reason='Unfilled market order converted at an aggressive quote'):
                    self.emit('CONVERSION_STARTED', order)
                    result = original(order, current_price)
                    self.emit('CONVERSION_COMPLETED', order, success=result.success, reason=result.message)
                    return result
            return call
        self.bind(engine.market_order_handler._conversion_service, 'convert_to_aggressive_limit', wrap_conversion)

        if recorder is not None:
            self.attach_recorder(recorder)
        return self

    def attach_recorder(self, recorder):
        def wrap_order_records(original):
            def call(round_number, orders, timestamp):
                start = len(recorder.order_data)
                result = original(round_number, orders, timestamp)
                for record, order in zip(recorder.order_data[start:], orders):
                    meta = self.meta[str(order.order_id)]
                    record.update({
                        'order_id': str(order.order_id), 'decision_id': meta['decision_id'],
                        'proposal_id': meta['proposal_id'],
                        'requested_quantity': meta['requested_quantity'],
                        'accepted_quantity': meta['accepted_quantity'],
                        'remaining_quantity': order.remaining_quantity,
                        'filled_quantity': order.filled_quantity, 'order_state': order.state.value,
                    })
                return result
            return call
        self.bind(recorder, '_record_order_data', wrap_order_records)

        def wrap_trade_records(original):
            def call(round_number, trades, timestamp):
                start = len(recorder.trade_data)
                result = original(round_number, trades, timestamp)
                for record, trade in zip(recorder.trade_data[start:], trades):
                    record.update({
                        'trade_id': trade.trade_id,
                        'buyer_order_id': str(trade.buyer_order_id),
                        'seller_order_id': str(trade.seller_order_id),
                    })
                return result
            return call
        self.bind(recorder, '_record_trade_data', wrap_trade_records)

    def snapshot(self, label='round_end'):
        self.emit('SNAPSHOT', label=label)
        for order in self.orders.orders.values():
            self.order_snapshots.append({'run_id': self.run_id, 'round_index': int(self.experiment_round_index),
                                         'label': label, **self.order_view(order)})
        for agent_id, resource in self.all_resources().items():
            self.agent_snapshots.append({'run_id': self.run_id, 'round_index': int(self.experiment_round_index),
                                         'label': label, 'agent_id': agent_id, **resource})

    def lifecycle_rows(self):
        executed = defaultdict(float)
        for trade in self.trades:
            executed[trade['buyer_order_id']] += float(trade['quantity'])
            executed[trade['seller_order_id']] += float(trade['quantity'])
        rows = []
        for oid, order in self.orders.orders.items():
            oid = str(oid)
            meta = self.meta[oid]
            accepted = bool(meta['accepted'])
            accepted_qty = float(meta['accepted_quantity'] or 0)
            filled = executed[oid]
            remaining = float(order.remaining_quantity)
            removed = float(meta['post_acceptance_removed_quantity'])
            terminal = order.state in (OrderState.CANCELLED, OrderState.FILLED)
            if not accepted:
                status = 'rejected'
            elif filled == 0:
                status = 'unfilled'
            elif abs(filled - accepted_qty) < 1e-8:
                status = 'fully_filled'
            else:
                status = 'partially_filled'
            residual = accepted_qty - filled - remaining - removed if accepted else None
            rows.append({
                'run_id': self.run_id, 'order_id': oid, 'decision_id': meta['decision_id'],
                'proposal_id': meta['proposal_id'], 'agent_id': str(order.agent_id),
                'side': order.side, 'order_type': order.order_type,
                'requested_quantity': meta['requested_quantity'], 'accepted': accepted,
                'accepted_quantity': accepted_qty, 'executed_quantity': filled,
                'unaccepted_quantity_reduction': float(meta['requested_quantity']) - accepted_qty if accepted else None,
                'post_acceptance_removed_quantity': removed,
                'remaining_quantity_raw': remaining,
                'remaining_open_quantity': remaining if accepted and not terminal else 0,
                'cancelled_remaining_quantity': remaining if accepted and order.state == OrderState.CANCELLED else 0,
                'execution_status': status, 'final_state': order.state.value,
                'validation_reason': meta['validation_reason'],
                'cancellation_reason': meta['cancellation_reason'],
                'replaces_order_ids': meta['replaces_order_ids'],
                'cash_commitment_final': order.current_cash_commitment,
                'share_commitment_final': order.current_share_commitment,
                'quantity_ledger_residual': residual,
                'quantity_ledger_ok': residual is None or abs(residual) < 1e-8,
                'terminal_commitments_released': (abs(order.current_cash_commitment) < 1e-8 and
                                                  abs(order.current_share_commitment) < 1e-8) if terminal else None,
            })
        return rows

    def validate(self):
        rows = self.lifecycle_rows()
        assert [row['event_seq'] for row in self.events] == list(range(1, len(self.events) + 1))
        decision_ids = {row['decision_id'] for row in self.decisions}
        assert len(decision_ids) == len(self.decisions)
        assert len({row['trade_id'] for row in self.trades}) == len(self.trades)
        for row in rows:
            assert row['decision_id'] in decision_ids
            assert row['quantity_ledger_ok'], row
            order = self.orders.get_order(row['order_id'])
            assert abs(float(order.filled_quantity) - row['executed_quantity']) < 1e-8, row
            if row['terminal_commitments_released'] is not None:
                assert row['terminal_commitments_released'], row
        by_id = {row['order_id']: row for row in rows}
        for trade in self.trades:
            assert by_id[trade['buyer_order_id']]['accepted']
            assert by_id[trade['seller_order_id']]['accepted']
        total_legs = sum(row['executed_quantity'] for row in rows)
        assert abs(total_legs - 2 * sum(float(t['quantity']) for t in self.trades)) < 1e-8
        for aid in self.agents.get_all_agent_ids():
            orders = self.orders.get_agent_orders(aid)
            resource = self.resource(aid)
            assert abs(sum(o.current_cash_commitment for o in orders) - resource['committed_cash']) < 1e-6
            assert abs(sum(o.current_share_commitment for o in orders) - resource['committed_shares']) < 1e-6
        return rows

    def export(self, recorder=None):
        rows = self.validate()
        flat = []
        for event in self.events:
            row = {key: value for key, value in event.items()
                   if key not in ('resource_before', 'resource_after', 'order', 'details')}
            for label in ('before', 'after'):
                resource = event['resource_' + label]
                if isinstance(resource, dict) and 'available_cash' in resource:
                    row.update({label + '_' + key: value for key, value in resource.items()})
            row['order_snapshot'] = event['order']
            row['details'] = event['details']
            row['resource_before_json'] = event['resource_before']
            row['resource_after_json'] = event['resource_after']
            flat.append(row)
        write_csv(self.output_dir / 'order_events.csv', flat)
        write_csv(self.output_dir / 'decision_orders.csv', self.proposals)
        write_csv(self.output_dir / 'order_lifecycle.csv', rows)
        write_csv(self.output_dir / 'trades_linked.csv', self.trades)
        write_csv(self.output_dir / 'order_snapshots.csv', self.order_snapshots)
        write_csv(self.output_dir / 'agent_snapshots.csv', self.agent_snapshots)
        write_csv(self.output_dir / 'rounds.csv', self.rounds)
        with (self.output_dir / 'decisions.jsonl').open('w', encoding='utf-8') as stream:
            for decision in self.decisions:
                stream.write(json.dumps(clean(decision), ensure_ascii=False) + '\n')
        if recorder is not None:
            write_csv(self.output_dir / 'recorded_orders.csv', recorder.order_data)
            write_csv(self.output_dir / 'recorded_trades.csv', recorder.trade_data)
        return rows


def attach_to_simulation(simulation, output_dir, run_id=None):
    '''Attach before simulation.run(); this function itself makes no model calls.

    Calling simulation.run() with LLM agents can make model calls. The notebook
    instead uses manual-response agents and blocks network access in the worker.
    '''
    if simulation.is_multi_stock:
        raise NotImplementedError('This research logger currently targets the single-stock Figure 6-7 setting')
    audit = LifecycleAudit(simulation.context, simulation.agent_repository,
                           simulation.order_repository, output_dir, run_id)
    audit.attach(simulation.decision_service, simulation.order_state_manager,
                 simulation.trade_execution_service, simulation.matching_engine,
                 recorder=simulation.data_recorder)
    simulation.register_before_round(lambda sim, r: audit.emit('ROUND_STARTED'))
    simulation.register_after_round(lambda sim, r: (audit.snapshot(), audit.export(sim.data_recorder)))
    return audit
"""
}
if not SOURCE_ROOT.exists():shutil.copytree(SOURCE_LOGGED,SOURCE_ROOT)
for relative,expected in EXPECTED_BASE_HASHES.items():
    if hashlib.sha256((SOURCE_ROOT/relative).read_text(encoding="utf-8").encode()).hexdigest()!=expected:
        raise RuntimeError(f"Base file in the Step 4 copy contains other changes: {relative}")
for relative,text in NEW_FILES.items():
    target=SOURCE_ROOT/relative
    if target.exists() and target.read_text(encoding="utf-8")!=text:
        raise RuntimeError(f"File contains other changes; overwrite stopped: {relative}")
    ast.parse(text,filename=relative)
for relative,text in NEW_FILES.items():(SOURCE_ROOT/relative).write_text(text,encoding="utf-8")
RUNNER=SOURCE_ROOT/"scripts/run_step4_longitudinal.py"
print("Step 4 runner and multi-round observer adapter written; syntax checks passed.")


## Code cell 4/9: Pre-generate shared dividend paths and the plan

For each seed, all prompt conditions share exactly the same 15 dividend values. Different seeds have different paths; matching does not depend on global random-number consumption during execution. Save paths and configuration before running experiments.


In [ ]:
command=[str(TEST_PYTHON),str(RUNNER),"prepare","--source",str(SOURCE_ROOT),"--output",str(PROTOCOL_DIR),
         "--model",API_MODEL,"--base-url",API_BASE_URL,"--seeds",json.dumps(SEEDS),
         "--prices",json.dumps(INITIAL_PRICES),"--arms",json.dumps(ARMS)]
prepared=subprocess.run(command,cwd=SOURCE_ROOT,capture_output=True,text=True,encoding="utf-8",env=child_env(),timeout=60)
if prepared.returncode:raise RuntimeError(prepared.stdout[-2000:]+prepared.stderr[-5000:])
PLAN_PATH=PROTOCOL_DIR/"experiment_plan.json";PLAN=read_json(PLAN_PATH)
assert PLAN["planned_model_requests"]==PLANNED_REQUESTS
EXPERIMENT_DIR=WORK_DIR/"experiments"/PLAN["experiment_id"]
API_CACHE_DIR=WORK_DIR/"model_response_cache"/PLAN["experiment_id"]
print(prepared.stdout.strip())
print("This is a request plan only; actual model calls in this cell: 0.")
print("Formal experiment directory: ",EXPERIMENT_DIR)


## Code cell 5/9: Inspect the plan and comparison scope

In formal multi-round runs, all agents observe their condition's evolving market history, so non-maker responses may change across conditions. This design compares the market's dynamic response and does not require identical non-maker orders across conditions.

Future dividend realizations are not disclosed in advance. Agents continue to receive model parameters and already-observed payments through the original information system.


In [ ]:
trials=read_csv(PROTOCOL_DIR/"planned_trials.csv")
paths=read_csv(PROTOCOL_DIR/"shared_dividend_paths.csv")
assert len(paths)==15*len(SEEDS) and len(trials)==len(SEEDS)*len(INITIAL_PRICES)*len(ARMS)
print("Planned market runs: ",len(trials),"; formal model requests planned: ",PLAN["planned_model_requests"])
display_table(trials,[("arm","Maker prompt condition"),("seed","Dividend-path and model seed"),("initial_price","Initial price")])
first_seed=str(SEEDS[0]);first_path=[r for r in paths if r["seed"]==first_seed]
display_table(first_path,[("market_round","Round"),("dividend_per_share","Planned dividend per share in this round")])
print("Identical seeds do not guarantee identical model outputs. Pair conditions by saving and actually using the same realized dividend sequence.")


## Code cell 6/9: Check the 15-round workflow offline

Use **explicitly scripted** buy, sell, and two-sided orders to check two configurations and one identical-input replay. These are three 15-round workflow checks with 0 model API calls. Scripted decisions are identical across the two conditions and do not test prompt effects.

Each round checks actual dividend payments, the cap, both counterparty order IDs, the quantity ledger, and commitments. Save synchronized quotes at four phases. Store the checks separately in `offline_checks`.


In [ ]:
OFFLINE_DIR=SESSION_DIR/"offline_checks"
if RUN_OFFLINE_CHECK:
    stream_command([str(TEST_PYTHON),str(RUNNER),"smoke","--source",str(SOURCE_ROOT),
                    "--plan",str(PLAN_PATH),"--output",str(OFFLINE_DIR)])
    OFFLINE_CHECKS=read_json(OFFLINE_DIR/"suite_checks.json")
    assert OFFLINE_CHECKS["status"]=="PASS"
    assert OFFLINE_CHECKS["new_model_requests_this_execution"]==0
    assert OFFLINE_CHECKS["research_results_available"] is False
    print("Offline multi-round checks passed. The formal model experiment has not been run.")
else:
    print("Offline checks skipped. Model requests in this cell: 0.")


## Code cell 7/9: Formal API experiment, skipped by default

Only enter the request flow when `api_experiment` is selected in cell 1.

- Use `OPENAI_API_KEY` or masked input. Do not write credentials into code or result archives.
- All 8 agents request one response per round. The current full plan has 1,080 requests; cached resumption can reduce new requests.
- Stop on failure. Save completed responses with their complete prompts, model, endpoint, and condition; replay completed portions from cache when resuming.
- Formal results and offline checks use separate directories; aggregation reads only formal results.

The original model snapshot is retained by default. Confirm team model/endpoint access. A model change creates a different experiment plan; all comparison conditions must use the same settings.


In [ ]:
API_RUN_COMPLETED=False
if RUN_MODE=="prepare_only":
    print("Preparation mode: skip the real model experiment. Model requests in this execution: 0.")
else:
    from getpass import getpass
    EXPERIMENT_DIR.mkdir(parents=True,exist_ok=True);API_CACHE_DIR.mkdir(parents=True,exist_ok=True)
    remaining=[]
    for trial in PLAN["trials"]:
        completed=EXPERIMENT_DIR/"runs"/trial["trial_key"]/"completed.json"
        prior=read_json(completed) if completed.exists() else {}
        if not prior or not (Path(prior["attempt_directory"])/"run_summary.json").is_file():remaining.append(trial)
    print("Market runs still to complete: ",len(remaining))
    print("Remaining requests before cache reuse: ",len(remaining)*15*8,"; new-request limit for this execution: ",MAX_NEW_API_REQUESTS)
    env=child_env()
    if remaining:
        key=os.environ.get("OPENAI_API_KEY") or getpass("Enter the API key for the configured endpoint (masked and not saved): ")
        if not key:raise RuntimeError("Formal mode requires a usable API configuration; preparation mode needs no key.")
        env["AHE_STEP4_API_KEY"]=key;del key
    command=[str(TEST_PYTHON),str(RUNNER),"run","--source",str(SOURCE_ROOT),"--plan",str(PLAN_PATH),
             "--output",str(EXPERIMENT_DIR),"--cache-dir",str(API_CACHE_DIR),
             "--max-new-requests",str(MAX_NEW_API_REQUESTS),"--allow-api"]
    try:stream_command(command,env=env)
    finally:env.pop("AHE_STEP4_API_KEY",None)
    ACTUAL_CHECKS=read_json(EXPERIMENT_DIR/"suite_checks.json")
    assert ACTUAL_CHECKS["status"]=="PASS" and ACTUAL_CHECKS["research_results_available"] is True
    API_RUN_COMPLETED=True
    print("Formal multi-round experiment completed. New requests in this execution: ",ACTUAL_CHECKS["new_model_requests_this_execution"])


## Code cell 8/9: Summarize completion and formal results

Show the research-result table only after the formal API experiment completes. Scripted offline orders do not enter that table.

Distinguish recorded price, actual within-round execution price, and synchronized two-sided midpoint. Leave execution price missing in no-trade rounds and midpoint missing with one-sided quotes. Weight execution-price errors by traded shares. Seeds/runs determine the number of experimental repetitions; individual orders are not independent experimental samples.


In [ ]:
if not API_RUN_COMPLETED:
    print("Step 4 progress: pre-generated dividend paths, saved the comparison plan, and completed offline multi-round checks if enabled.")
    print("No new Step 4 model-experiment results are available. The formal experiment requires an API.")
else:
    results=read_csv(EXPERIMENT_DIR/"trial_summary.csv")
    assert len(results)==len(PLAN["trials"])
    for row in results:
        assert row["mode"]=="api_experiment" and int(row["rounds_completed"])==15
        assert row["dividend_path_checks"]==row["order_trade_resource_checks"]=="PASS"
        assert int(row["scripted_fixture_response_count"])==0
    display_table(results,[("arm","Prompt condition"),("seed","seed"),("maker_proposed_two_sided","Two-sided proposals / 30"),
        ("maker_accepted_two_sided","Both sides accepted / 30"),("trade_count","Trades"),("executed_shares","Executed shares"),
        ("final_recorded_price","Final recorded price"),("last_actual_trade_price","Last actual execution price"),
        ("zero_trade_rounds","No-trade rounds")])
    print("Round data: round_metrics.csv in each trial directory.")
    print("Synchronized quotes: quote_snapshots.csv; actual within-round dividends: dividend_realizations.csv.")
    print("The original simulator's market_data dividend column may refer to the previous payment; verify this round's path using the actual-payment file.")
    print("Scope: short-run dynamic comparisons under shared dividend paths; small-sample results do not establish systemic-risk conclusions.")


## Code cell 9/9: Package results and save completion status

Preparation mode exports `STEP4_PREPARATION.zip`; a completed formal run exports `STEP4_API_RESULTS.zip`. Include the plan, code, and checks. Formal archives additionally contain round results, synchronized quotes, actual dividends, and linked order logs. Exclude credentials.

Press `Ctrl + S` to save the notebook, then export HTML. Preparation-mode HTML documents completed preparation and workflow checks only.


In [ ]:
state={"mode":RUN_MODE,"api_experiment_completed":API_RUN_COMPLETED,
       "experiment_id":PLAN["experiment_id"],"planned_model_requests":PLAN["planned_model_requests"],
       "progress_scope":"Completed new 15-round model experiments" if API_RUN_COMPLETED else "Experiment preparation and offline plumbing checks only"}
(SESSION_DIR/"completion_state.json").write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding="utf-8")
suffix="STEP4_API_RESULTS" if API_RUN_COMPLETED else "STEP4_PREPARATION"
ZIP_PATH=SESSION_DIR.with_name(SESSION_DIR.name+"_"+suffix+".zip")
with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as package:
    for path in SESSION_DIR.rglob("*"):
        if path.is_file() and "__pycache__" not in path.parts:package.write(path,"session/"+path.relative_to(SESSION_DIR).as_posix())
    for relative in NEW_FILES:package.write(SOURCE_ROOT/relative,"code/"+relative)
    if API_RUN_COMPLETED:
        for name in ("trial_summary.csv","suite_checks.json"):package.write(EXPERIMENT_DIR/name,"api_results/"+name)
        for row in read_csv(EXPERIMENT_DIR/"trial_summary.csv"):
            attempt=Path(row["attempt_directory"])
            for path in attempt.rglob("*"):
                if path.is_file():package.write(path,"api_results/runs/"+row["trial_key"]+"/"+path.relative_to(attempt).as_posix())
print("Run mode: ",RUN_MODE)
print("Formal model experiment completed: ",API_RUN_COMPLETED)
print("Currently reportable scope: ",state["progress_scope"])
print("Result archive: ",ZIP_PATH)
print("Press Ctrl + S to save, then export HTML to retain your execution results.")
